# 数据模型

调用特殊方法来执行基本对象操作。特殊方法供Python解释器调用，而非开发者。

在处理内置类型时，Python直接读取了其C底层实现的结构体PyVarObject的ob_size字段。该字段保存着容器的项数。

特殊方法很多时候都是隐式调用的,一般不直接调用，除非涉及大量的元编程。即便如此，大部分时间也只是实现特殊方法，很少显式调用。唯一例外的是__init__方法，为自定义类实现__init__方法时经常直接调用它来调用超类的初始化方法。如果需要调用特殊方法，则最好调用相应的内置函数。

常用需要实现特殊方法的语言结构
- 容器
- 属性存取
- 迭代
- 运算符重载
- 函数和方法调用
- 字符串表示和格式化
- 使用await异步编程
- 对象创建和析构
- 使用with或async with语句管理上下文

数据模型和组合模式

In [11]:
import collections
from random import choices

Card = collections.namedtuple("Card", ["rank", "suit"])


class FrenchDeck:
    ranks = [str(n) for n in range(2, 11)] + list("JQKA")
    suits = "spades diamonds clubs hearts".split()

    def __init__(self):
        self._cards = [Card(rank, suit) for suit in self.suits for rank in self.ranks]

    def __len__(self):
        return len(self._cards)

    def __getitem__(self, position):
        return self._cards[position]


beer_card = Card("7", "hearts")
print(beer_card)

deck = FrenchDeck()
print(beer_card in deck)
# print(Card('beats', 'A') in deck)

print(len(deck))
# deck[53]
# print(deck[0])
# print(deck[-1])
checked = choices(deck)
print(checked)

# print(deck[:3])

# print(deck[12::13])

# for card in deck:
#   print(card)

# for card in reversed(deck):
#   print(card)

suit_values = dict(spades=3, hearts=2, diamonds=1, clubs=0)


def spades_high(card):
    rank_value = FrenchDeck.ranks.index(card.rank)
    return rank_value * len(suit_values) + suit_values[card.suit]


for card in sorted(deck, key=spades_high):
    print(card)

Card(rank='7', suit='hearts')
True
52
[Card(rank='7', suit='hearts')]
Card(rank='2', suit='clubs')
Card(rank='2', suit='diamonds')
Card(rank='2', suit='hearts')
Card(rank='2', suit='spades')
Card(rank='3', suit='clubs')
Card(rank='3', suit='diamonds')
Card(rank='3', suit='hearts')
Card(rank='3', suit='spades')
Card(rank='4', suit='clubs')
Card(rank='4', suit='diamonds')
Card(rank='4', suit='hearts')
Card(rank='4', suit='spades')
Card(rank='5', suit='clubs')
Card(rank='5', suit='diamonds')
Card(rank='5', suit='hearts')
Card(rank='5', suit='spades')
Card(rank='6', suit='clubs')
Card(rank='6', suit='diamonds')
Card(rank='6', suit='hearts')
Card(rank='6', suit='spades')
Card(rank='7', suit='clubs')
Card(rank='7', suit='diamonds')
Card(rank='7', suit='hearts')
Card(rank='7', suit='spades')
Card(rank='8', suit='clubs')
Card(rank='8', suit='diamonds')
Card(rank='8', suit='hearts')
Card(rank='8', suit='spades')
Card(rank='9', suit='clubs')
Card(rank='9', suit='diamonds')
Card(rank='9', suit='h

In [24]:
# 模拟数值类型
"""
vector2d.py

add:
v1 = Vector(2, 4)
v2 = Vector(2, 1)
v1 + v2

"""

import math


class Vector:
    def __init__(self, x=0, y=0):
        self.x = x
        self.y = y

    def __repr__(self):
        return f"Vector({self.x!r}, {self.y!r})"

    def __abs__(self):
        return math.hypot(self.x, self.y)

    def __bool__(self):
        return bool(abs(self))

    def __add__(self, other):
        x = self.x + other.x
        y = self.y + other.y
        return Vector(x, y)

    def __mul__(self, scalar):
        return Vector(self.x * scalar, self.y * scalar)


v1 = Vector(2, 4)
v2 = Vector(1, 3)
print(v1 + v2)

v3 = Vector(3, 4)
print(abs(v3))

print(v3 * 2)

Vector(3, 7)
5.0
Vector(6, 8)


In [22]:
from collections import abc, deque

print(issubclass(tuple, abc.Sequence))
print(issubclass(list, abc.Sequence))
print(issubclass(str, abc.Sequence), issubclass(str, abc.MutableSequence))
print(issubclass(deque, abc.Sequence), issubclass(deque, abc.MutableSequence))

True
True
True False
True True


## 序列模式匹配
match/case 支持析构
匹配对象需要同时满足条件:
- 匹配对象是序列
- 匹配对象和模式的项数相同
- 对应项相互匹配，包括嵌套的项

序列模式可以写成元组或列表，或任意形式的嵌套元组或列表。
在序列模式中，[]和()的意思是一样的。
序列模式可以匹配collection.abc.Sequence的多数实际子类或虚拟子类的实例，但str、bytes和bytearray除外。标准库中的以下类型与序列模式兼容: list、memoryview、array.array、tuple、range、collections.deque。模式不析构序列以外的可迭代对象。模式中_匹配相应位置上的任意一项，但不绑定匹配项的值。可以在模式中多次出现。模式支持类型信息匹配。

In [ ]:
# *_ 匹配任意数量的项，而且不绑定变量
case[str(name), *_, (float(lat), float(lon))]

## 切片
为什么切片和区间排除最后一项？
- 在仅指定停止位置时，容易判断切片和区间长度
- 同时指定起始和停止位置时，容易计算切片和区间长度
- 方便在某个位置将序列拆分成不重叠的两部分


s[a:b:c] 步距c可以是负数，反向返回项
a:b:c表示法只在[]内部有效，表示索引或下标运算符。实际调用seq.__getitem__(slice(a,b,c))。

切片对象
多维切片和省略号
除memoryview外，python内置的序列类型都是一维的，因此只支持一个索引或切片，不支持索引或切片元组。


In [29]:
s = [1, 2, 3, 4, 5, 6]
sl = slice(1, 3, 1)
print(type(sl))
print(sl)

<class 'slice'>
slice(1, 3, 1)


切片赋值
在赋值语句左侧使用切片表示法，或者作为del语句的目标，可以就地移植、切除或以其他方式改变序列。
切片赋值，右侧必须是一个可迭代对象。 

In [32]:
l = list(range(10))
l[2:5] = [20, 30]
print(l)
del l[5:10:2]
print(l)
l[3::2] = [11, 12]
print(l)

[0, 1, 20, 30, 5, 6, 7, 8, 9]
[0, 1, 20, 30, 5, 7, 9]
[0, 1, 20, 11, 5, 12, 9]


序列中的+和*运算符

序列的`+`运算中的两个对象必须是同一种而且不可修改的序列。
`+`和`*`运算始终创建一个对象，不更改操作数。

In [35]:
board = 3 * [["_"] * 3]
print(board)
board[1][2] = "X"
print(board)

[['_', '_', '_'], ['_', '_', '_'], ['_', '_', '_']]
[['_', '_', 'X'], ['_', '_', 'X'], ['_', '_', 'X']]


In [38]:
board2 = [["_"] * 3 for i in range(3)]
print(board2)
board2[1][2] = "X"
print(board2)

[['_', '_', '_'], ['_', '_', '_'], ['_', '_', '_']]
[['_', '_', '_'], ['_', '_', 'X'], ['_', '_', '_']]


增量赋值运算符`+=`和`*=`

`+=`运算背后的执行特殊方法__iadd__,如果没有实现，则调用__add__。
增量赋值不是原子操作。
不要在元组中存放可变的项。

In [40]:
l = [1, 2, 3]
print(id(l))
l *= 2
print(id(l))
print(l)
l[0] = 11
print(l)

4480636032
4480636032
[1, 2, 3, 1, 2, 3]
[11, 2, 3, 1, 2, 3]


In [41]:
t = (1, 2, 3)
print(id(t))
t *= 2
print(id(t))
print(t)

4479498880
4480900864
(1, 2, 3, 1, 2, 3)


In [42]:
t = (1, 2, [30, 40])
t[2] += [50, 60]
print(t)

TypeError: 'tuple' object does not support item assignment

In [44]:
import dis

dis.dis("s[a] += b")

  0           0 RESUME                   0

  1           2 LOAD_NAME                0 (s)
              4 LOAD_NAME                1 (a)
              6 COPY                     2
              8 COPY                     2
             10 BINARY_SUBSCR
             14 LOAD_NAME                2 (b)
             16 BINARY_OP               13 (+=)
             20 SWAP                     3
             22 SWAP                     2
             24 STORE_SUBSCR
             28 RETURN_CONST             0 (None)


list.sort方法就地排序列表，返回None。
就地更改对象的函数或方法应该返回None,让调用方清楚地知道接收者已被更改，没有创建新的对象。
内置sorted函数接收任何可迭代对象作为参数，包括不可变序列和生成器，返回创建的新列表。
list.sort和sorted均接受两个可选的关键字参数。
- reverse 默认值False
- key 应用到每一项的函数

In [52]:
fruits = ["grape", "raspberry", "apple", "banana"]
sorted_fruits = sorted(fruits)
print(fruits)
print(sorted_fruits)
sorted_fruits2 = sorted(fruits, reverse=True)
print(sorted_fruits2)
sorted_fruits3 = sorted(fruits, key=len)
print(sorted_fruits3)
fruits.sort()
print(fruits)

fruits.sort(key=len, reverse=True)
print(fruits)

['grape', 'raspberry', 'apple', 'banana']
['apple', 'banana', 'grape', 'raspberry']
['raspberry', 'grape', 'banana', 'apple']
['grape', 'apple', 'banana', 'raspberry']
['apple', 'banana', 'grape', 'raspberry']
['raspberry', 'banana', 'apple', 'grape']


标准库中bisect模块提供一种二进制搜索算法，其中bisect.insort函数还可确保排序的序列始终保持不变。

## 数组
array.array只包含数值。不允许向数组中添加与指定类型不同的值。支持所有可变序列操作(包括pop、insert、extend),还有快速加载和保存项的方法，如.frombytes和.tofile。

In [54]:
from array import array
from random import random

floats = array("d", (random() for i in range(10**2)))
print(floats[-1])
fp = open('floats.bin', 'wb')
floats.tofile(fp)
fp.close()


0.5720945877281837


In [60]:
from array import array

fp = open('floats.bin', 'rb')
floats2 = array('d')
floats2.fromfile(fp, 10**2)
fp.close()
print(floats2[-1])
print(floats == floats2)

0.5720945877281837
True
